# Proje Tanımı
## Projenin Amacı

Bu proje; eksik veriler, groupby, concat, merge, ileri pandas operasyonları ve Excel ile çalışma konularını kullanarak **gerçekçi bir satış–müşteri analiz sistemi** kurmayı hedefler.

## Pandas Neden Tercih Edilir?

Pandas, tablo formatındaki verileri hızlı bir şekilde temizlemek, birleştirmek ve analiz etmek için kullanılır. Eksik değerlerle çalışma, gruplama yaparak özet metrik çıkarma ve farklı kaynaklardan gelen tabloları bir araya getirme gibi işlemleri pratik hale getirir. Bu nedenle gerçek iş senaryolarında en çok kullanılan veri analiz araçlarından biridir.

## Bu Projede Kapsanan Pandas Konuları
* Eksik veriler: isna, fillna, dropna
* Birleştirme: merge (relation mantığı)
* Gruplama: groupby, agg
* Veri birleştirme: concat
* İleri işlemler: koşullu filtreleme, yeni sütun, apply
* Excel çıktısı: to_excel



In [1]:
# 2. Creating Datasets
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "CustomerID": [1,2,3,4,5,6,7,8],
    "CustomerName": ["Ahmet","Ayşe","Mehmet","Zeynep","Can","Elif","Burak","Seda"],
    "City": ["İstanbul","Ankara","İzmir","Bursa","Antalya","İstanbul","Ankara","İzmir"],
    "Age": [25, np.nan, 40, 35, np.nan, 29, 50, 31]
})

sales = pd.DataFrame({
    "SaleID": [101,102,103,104,105,106,107,108,109],
    "CustomerID": [1,2,3,4,5,6,7,8,1],
    "Product": ["Laptop","Handy","Tablet","Laptop","Headset","Tablet","Handy","Laptop","Tablet"],
    "Amount": [15000, 8000, np.nan, 12000, 1500, 6000, 9000, np.nan, 7000]
})

regions = pd.DataFrame({
    "City": ["İstanbul","Ankara","İzmir","Bursa","Antalya"],
    "Region": ["Marmara","İç Anadolu","Ege","Marmara","Akdeniz"]
})

In [2]:
# 3. NaN Values
customer_isna = customers.isna().sum()
sales_isna = sales.isna().sum()
regions_isna = regions.isna().sum()

avr_age = customers["Age"].mean()

customers["Age"] = customers["Age"].fillna(avr_age)

clean_sales = sales.dropna(subset=["Amount"])

print(customer_isna)
print(sales_isna)
print(regions_isna)
print(f"Average Age: \n{avr_age}")
print(f"Updated Sales: \n{clean_sales}")

CustomerID      0
CustomerName    0
City            0
Age             2
dtype: int64
SaleID        0
CustomerID    0
Product       0
Amount        2
dtype: int64
City      0
Region    0
dtype: int64
Average Age: 
35.0
Updated Sales: 
   SaleID  CustomerID  Product   Amount
0     101           1   Laptop  15000.0
1     102           2    Handy   8000.0
3     104           4   Laptop  12000.0
4     105           5  Headset   1500.0
5     106           6   Tablet   6000.0
6     107           7    Handy   9000.0
8     109           1   Tablet   7000.0


In [3]:
# 4. Merge
merged_df = pd.merge(customers, clean_sales, on="CustomerID", how="inner")
merged_final_frame = pd.merge(merged_df, regions, on="City", how="left")
merged_final_frame

,CustomerID,CustomerName,City,Age,SaleID,Product,Amount,Region
0,1,Ahmet,İstanbul,25.0,101,Laptop,15000.0,Marmara
1,1,Ahmet,İstanbul,25.0,109,Tablet,7000.0,Marmara
2,2,Ayşe,Ankara,35.0,102,Handy,8000.0,İç Anadolu
3,4,Zeynep,Bursa,35.0,104,Laptop,12000.0,Marmara
4,5,Can,Antalya,35.0,105,Headset,1500.0,Akdeniz
5,6,Elif,İstanbul,29.0,106,Tablet,6000.0,Marmara
6,7,Burak,Ankara,50.0,107,Handy,9000.0,İç Anadolu


In [4]:
# 5. GroupBy Operations
city_based_total_sales = merged_final_frame.groupby("City")["Amount"].sum()
region_based_total_sales = merged_final_frame.groupby("Region")["Amount"].mean()
region_based_total_sales

Region
Akdeniz        1500.0
Marmara       10000.0
İç Anadolu     8500.0
Name: Amount, dtype: float64

In [5]:
# 6. Concatenation
sales_january = pd.DataFrame({
    "SaleID": [201,202,203,204],
    "CustomerID": [1,1,2,3],
    "Product": ["Laptop","Handy","Tablet","Laptop"],
    "Amount": [15000, 8000, np.nan, 12000]
})

sales_february = pd.DataFrame({
    "SaleID": [205,206,207,208],
    "CustomerID": [2,3,4,5],
    "Product": ["Laptop","Handy","Tablet","Laptop"],
    "Amount": [15000, 8000, np.nan, 12000]
})

all_sales = pd.concat([sales_january, sales_february],axis=0)

all_sales

,SaleID,CustomerID,Product,Amount
0,201,1,Laptop,15000.0
1,202,1,Handy,8000.0
2,203,2,Tablet,NaN
3,204,3,Laptop,12000.0
0,205,2,Laptop,15000.0
1,206,3,Handy,8000.0
2,207,4,Tablet,NaN
3,208,5,Laptop,12000.0


In [6]:
# 7. Advanced Pandas Operations
customers_older_than_30 = merged_final_frame[merged_final_frame["Age"]>30]  #Customers older than 30 years old.
merged_final_frame["Amount_With_Tax"] = merged_final_frame["Amount"] * 1.18 #Price with taxees

def apply_discount(price):
    discounted_price = price * 0.90
    return discounted_price

merged_final_frame["%10 Discount"] = merged_final_frame["Amount_With_Tax"].apply(apply_discount)

print(f"Customers Older Than 30: \n{customers_older_than_30}")
print(f"%10 Discount: \n{merged_final_frame}")

Customers Older Than 30: 
   CustomerID CustomerName     City   Age  SaleID  Product   Amount  \
2           2         Ayşe   Ankara  35.0     102    Handy   8000.0   
3           4       Zeynep    Bursa  35.0     104   Laptop  12000.0   
4           5          Can  Antalya  35.0     105  Headset   1500.0   
6           7        Burak   Ankara  50.0     107    Handy   9000.0   

       Region  
2  İç Anadolu  
3     Marmara  
4     Akdeniz  
6  İç Anadolu  
%10 Discount: 
   CustomerID CustomerName      City   Age  SaleID  Product   Amount  \
0           1        Ahmet  İstanbul  25.0     101   Laptop  15000.0   
1           1        Ahmet  İstanbul  25.0     109   Tablet   7000.0   
2           2         Ayşe    Ankara  35.0     102    Handy   8000.0   
3           4       Zeynep     Bursa  35.0     104   Laptop  12000.0   
4           5          Can   Antalya  35.0     105  Headset   1500.0   
5           6         Elif  İstanbul  29.0     106   Tablet   6000.0   
6           7      

In [8]:
# 8. Exporting Clean data into Excel
merged_final_frame.to_excel("final_sales_report.xlsx", index=False)

In [15]:
# 9. Dashboard
print(f"Total Sale: {merged_final_frame['Amount'].sum()}")
print(f"Most Profitable City: {city_based_total_sales.idxmax()}")
print(f"Most Powerful City: {city_based_total_sales.idxmax()}")

Total Sale: 58500.0
Most Profitable City: İstanbul
Most Powerful City: İstanbul
